<a href="https://colab.research.google.com/github/NadunPrabhasha/SE4050-Deep-Learning-Final-Assignment/blob/main/Banana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!ls


sample_data


In [2]:
# ResNet50 — Banana Leaf Disease classification (Colab)
- Training pipeline (transfer learning + fine-tune)
- Training & validation graphs
- Grad-CAM visualization to see model attention


SyntaxError: invalid syntax (137402808.py, line 2)

In [ ]:
import sys, tensorflow as tf
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

# If GPU not present, go to Runtime -> Change runtime type -> GPU


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create folder to save models/outputs
!mkdir -p /content/drive/MyDrive/banana_leaf_models


In [ ]:
# Colab usually has TF installed — this ensures matplotlib present
!pip install -q matplotlib


In [ ]:
# This cell tries a few common locations, and if not present prompts manual upload.
import os, zipfile
content_zip = "/content/banana_leaf_disease.zip"
drive_zip = "/content/drive/MyDrive/banana_leaf_disease.zip"
extract_path = "/content/banana_leaf_disease"

if os.path.exists(extract_path):
    print("Dataset already extracted at", extract_path)
else:
    if os.path.exists(content_zip):
        zip_path = content_zip
    elif os.path.exists(drive_zip):
        zip_path = drive_zip
    else:
        # Ask user to upload
        from google.colab import files
        print("Please upload banana_leaf_disease.zip (use the dialog).")
        uploaded = files.upload()
        # pick first uploaded file
        zip_path = None
        for name in uploaded.keys():
            if name.lower().endswith('.zip'):
                zip_path = os.path.join('/content', name)
                break
        if zip_path is None:
            raise FileNotFoundError("No .zip uploaded. Upload banana_leaf_disease.zip and re-run this cell.")

    print("Using zip file:", zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print("Extracted to:", extract_path)

# Quick directory listing
print("Top-level folders:", os.listdir(extract_path))


In [ ]:
import os
base = "/content/banana_leaf_disease"
classes = sorted([d for d in os.listdir(base) if os.path.isdir(os.path.join(base, d))])
counts = {}
total = 0
for c in classes:
    n = len([f for f in os.listdir(os.path.join(base,c)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    counts[c] = n
    total += n

print("Classes:", classes)
print("Counts per class:", counts)
print("Total images:", total)


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.15,
    width_shift_range=0.12,
    height_shift_range=0.12,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

train_gen = datagen.flow_from_directory(
    "/content/banana_leaf_disease",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED
)

val_gen = datagen.flow_from_directory(
    "/content/banana_leaf_disease",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED
)

print("Class indices:", train_gen.class_indices)
NUM_CLASSES = len(train_gen.class_indices)


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
# Freeze base
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)

model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

checkpoint_path = "/content/drive/MyDrive/banana_leaf_models/resnet50_best.h5"
checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
early = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1)

callbacks = [checkpoint, reduce_lr, early]


In [ ]:
EPOCHS_HEAD = 12

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks
)


In [ ]:
# Unfreeze last N layers
N = 30
for layer in base_model.layers[-N:]:
    layer.trainable = True

# Recompile with lower LR
model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

EPOCHS_FINE = 6
fine_history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FINE,
    callbacks=callbacks
)


In [ ]:
# load best saved checkpoint (if exists)
import os
if os.path.exists(checkpoint_path):
    model.load_weights(checkpoint_path)
    print("Loaded best weights from checkpoint.")
else:
    print("No checkpoint found at", checkpoint_path)

final_save_path = "/content/drive/MyDrive/banana_leaf_models/resnet50_final.h5"
model.save(final_save_path)
print("Saved final model to:", final_save_path)

# Save class indices mapping
import json
class_map_path = "/content/drive/MyDrive/banana_leaf_models/class_indices.json"
with open(class_map_path, 'w') as f:
    json.dump(train_gen.class_indices, f, indent=2)
print("Saved class indices to:", class_map_path)


In [ ]:
import matplotlib.pyplot as plt

def plot_train_val(h1, h2=None, title=""):
    # Collect metrics
    acc = h1.history.get('accuracy', [])
    val_acc = h1.history.get('val_accuracy', [])
    loss = h1.history.get('loss', [])
    val_loss = h1.history.get('val_loss', [])

    if h2:
        acc += h2.history.get('accuracy', [])
        val_acc += h2.history.get('val_accuracy', [])
        loss += h2.history.get('loss', [])
        val_loss += h2.history.get('val_loss', [])

    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(epochs, acc, label='train acc')
    plt.plot(epochs, val_acc, label='val acc')
    plt.title(f'{title} Accuracy')
    plt.xlabel('Epoch')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(epochs, loss, label='train loss')
    plt.plot(epochs, val_loss, label='val loss')
    plt.title(f'{title} Loss')
    plt.xlabel('Epoch')
    plt.legend()

    plt.show()

plot_train_val(history, fine_history, title="ResNet50")


In [ ]:
loss, acc = model.evaluate(val_gen)
print(f"Validation loss: {loss:.4f}  |  Validation accuracy: {acc:.4f}")


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import os

def predict_and_show(img_path, model, class_indices, IMG_SIZE=(224,224)):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    plt.imshow(img); plt.axis('off'); plt.title("Input")
    x = image.img_to_array(img)/255.0
    x = np.expand_dims(x, axis=0)
    pred = model.predict(x)[0]
    pred_idx = int(np.argmax(pred))
    inv_map = {v:k for k,v in class_indices.items()}
    print("Image:", img_path)
    print("Predicted:", inv_map[pred_idx], "  Confidence:", float(pred[pred_idx]))
    return x, pred, pred_idx

# Example: pick a sample image automatically and predict
base_dir = "/content/banana_leaf_disease"
classes = sorted(os.listdir(base_dir))
sample_class = classes[0]
files = [f for f in os.listdir(os.path.join(base_dir, sample_class)) if f.lower().endswith(('.jpg','.jpeg','.png'))]
sample_path = os.path.join(base_dir, sample_class, files[0])
predict_and_show(sample_path, model, train_gen.class_indices, IMG_SIZE)


In [ ]:
import os
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import numpy as np

# ✅ Automatically detect where the actual class folders are
base_dir = "/content/banana_leaf_disease"

# If your dataset has an extra folder layer, detect it
subdirs = [os.path.join(base_dir, d) for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
if len(subdirs) == 1 and any(os.path.isdir(os.path.join(subdirs[0], d)) for d in os.listdir(subdirs[0])):
    base_dir = subdirs[0]

print("✅ Using dataset folder:", base_dir)

# ✅ Get valid class folders that contain at least one image
classes = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))])
classes_with_images = []
for c in classes:
    folder = os.path.join(base_dir, c)
    files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(files) > 0:
        classes_with_images.append(c)

if not classes_with_images:
    raise RuntimeError("❌ No class folders with images found in " + base_dir)

sample_class = classes_with_images[0]
sample_folder = os.path.join(base_dir, sample_class)
files = [f for f in os.listdir(sample_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
sample_path = os.path.join(sample_folder, files[0])

print("✅ Sample class:", sample_class)
print("✅ Using image:", sample_path)

def predict_and_show(img_path, model, class_indices, IMG_SIZE=(224,224)):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    plt.imshow(img); plt.axis('off'); plt.title("Input Image")
    x = image.img_to_array(img)/255.0
    x = np.expand_dims(x, axis=0)
    pred = model.predict(x)[0]
    pred_idx = int(np.argmax(pred))
    inv_map = {v:k for k,v in class_indices.items()}
    print("Predicted:", inv_map[pred_idx], "Confidence:", float(pred[pred_idx]))
    return x, pred, pred_idx

x, pred, pred_idx = predict_and_show(sample_path, model, train_gen.class_indices, IMG_SIZE)


In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

def get_last_conv_layer(model):
    # Find the last conv layer automatically
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError("No Conv2D layer found in the model")

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    img_array: (1, H, W, 3), preprocessed (rescaled 0-1 as we used)
    model: compiled model
    last_conv_layer_name: name of convolutional layer to target
    pred_index: index of class to compute Grad-CAM for (if None, takes top pred)
    returns: heatmap (H, W) normalized 0..1
    """
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)

    # Compute channel-wise mean of gradients
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalize
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    heatmap = heatmap.numpy()
    heatmap = cv2.resize(heatmap, (IMG_SIZE[1], IMG_SIZE[0]))
    return heatmap

def overlay_heatmap_on_image(img_path_or_array, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    # Load original image
    if isinstance(img_path_or_array, str):
        img = cv2.imread(img_path_or_array)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE[1], IMG_SIZE[0]))
    else:
        # img_path_or_array is array scaled 0..1
        img = np.squeeze(img_path_or_array) * 255.0
        img = img.astype(np.uint8)
        img = cv2.resize(img, (IMG_SIZE[1], IMG_SIZE[0]))

    heatmap_uint8 = np.uint8(255 * heatmap)
    colored_map = cv2.applyColorMap(heatmap_uint8, colormap)
    colored_map = cv2.cvtColor(colored_map, cv2.COLOR_BGR2RGB)

    superimposed = colored_map * alpha + img
    superimposed = np.clip(superimposed / superimposed.max() * 255.0, 0, 255).astype(np.uint8)
    return img.astype(np.uint8), colored_map, superimposed


In [ ]:
# Pick sample image (you can replace path)
base_dir = "/content/banana_leaf_disease"
classes = sorted(os.listdir(base_dir))
sample_class = classes[0]
files = [f for f in os.listdir(os.path.join(base_dir, sample_class)) if f.lower().endswith(('.jpg','.jpeg','.png'))]
sample_path = os.path.join(base_dir, sample_class, files[0])
print("Sample image:", sample_path)

# Prepare image array (match preprocessing used during train: rescale 1/255)
from tensorflow.keras.preprocessing import image
img = image.load_img(sample_path, target_size=IMG_SIZE)
img_array = image.img_to_array(img) / 255.0
input_array = np.expand_dims(img_array, axis=0)

# Get last conv layer name automatically
last_conv = get_last_conv_layer(model)
print("Using last conv layer:", last_conv)

# Predict and get class index
preds = model.predict(input_array)
pred_idx = int(np.argmax(preds[0]))
inv_map = {v:k for k,v in train_gen.class_indices.items()}
print("Predicted:", inv_map[pred_idx], " confidence:", float(preds[0][pred_idx]))

# Make heatmap
heatmap = make_gradcam_heatmap(input_array, model, last_conv, pred_index=pred_idx)

# Overlay
orig, cmap, superimposed = overlay_heatmap_on_image(sample_path, heatmap, alpha=0.4)

# Display results
plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(orig); plt.title("Original"); plt.axis('off')
plt.subplot(1,3,2); plt.imshow(cmap); plt.title("Heatmap"); plt.axis('off')
plt.subplot(1,3,3); plt.imshow(superimposed); plt.title("Grad-CAM Overlay"); plt.axis('off')
plt.show()


In [ ]:
import os
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import numpy as np

# 🔍 Auto-detect correct dataset directory
base_dir = "/content/banana_leaf_disease"

# If dataset is nested inside another folder, fix the path
subdirs = [os.path.join(base_dir, d) for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
if len(subdirs) == 1 and any(os.path.isdir(os.path.join(subdirs[0], d)) for d in os.listdir(subdirs[0])):
    base_dir = subdirs[0]

print("✅ Using dataset folder:", base_dir)

# ✅ Find a class folder that actually has images
classes = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
found_image = False

for c in classes:
    folder = os.path.join(base_dir, c)
    files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(files) > 0:
        sample_class = c
        sample_path = os.path.join(folder, files[0])
        found_image = True
        break

if not found_image:
    raise FileNotFoundError("❌ No image files found in any class folder under: " + base_dir)

print(f"✅ Sample class: {sample_class}")
print(f"✅ Using image: {sample_path}")

# === Prediction helper function ===
def predict_and_show(img_path, model, class_indices, IMG_SIZE=(224,224)):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    plt.imshow(img)
    plt.axis('off')
    plt.title("Input Image")

    x = image.img_to_array(img)/255.0
    x = np.expand_dims(x, axis=0)

    pred = model.predict(x)[0]
    pred_idx = np.argmax(pred)
    inv_map = {v:k for k,v in class_indices.items()}
    print("Predicted:", inv_map[pred_idx], "| Confidence:", float(pred[pred_idx]))

    return x, pred, pred_idx

# Run prediction
x, pred, pred_idx = predict_and_show(sample_path, model, train_gen.class_indices, IMG_SIZE)


In [ ]:
import matplotlib.pyplot as plt
rows = []
for cls in classes:
    files = [f for f in os.listdir(os.path.join(base, cls)) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if not files:
        continue
    p = os.path.join(base, cls, files[0])
    img = image.load_img(p, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)/255.0
    input_array = np.expand_dims(img_array, axis=0)
    preds = model.predict(input_array)
    pred_idx = int(np.argmax(preds[0]))
    heatmap = make_gradcam_heatmap(input_array, model, last_conv, pred_index=pred_idx)
    _, _, overlay = overlay_heatmap_on_image(p, heatmap, alpha=0.4)
    rows.append((cls, overlay, preds[0][pred_idx]))

# Plot grid
n = len(rows)
plt.figure(figsize=(4*n,4))
for i,(cls, img, conf) in enumerate(rows):
    plt.subplot(1,n,i+1)
    plt.imshow(img)
    plt.title(f"{cls}\nconf:{conf:.2f}")
    plt.axis('off')
plt.show()


In [ ]:
import os

# Root folder (where you unzipped)
base = "/content/banana_leaf_disease"

# --- Auto-detect correct subdirectory ---
subdirs = [os.path.join(base, d) for d in os.listdir(base) if os.path.isdir(os.path.join(base, d))]
# If there is only one folder inside and that folder itself has subfolders, go one level deeper
if len(subdirs) == 1 and any(os.path.isdir(os.path.join(subdirs[0], d)) for d in os.listdir(subdirs[0])):
    base = subdirs[0]

# Show what we’re using
print("✅ Dataset base folder:", base)

# Check which class folders exist
classes = [d for d in os.listdir(base) if os.path.isdir(os.path.join(base, d))]
print("✅ Detected classes:", classes)

# Verify that they contain images
for cls in classes:
    path = os.path.join(base, cls)
    images = [f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Class '{cls}' → {len(images)} images found")
